In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 5.1 - Full Pipeline (Load → PSF Soft Labels → Plot)
Physics-consistent with batch_downsampling_pipeline v1.2.0.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter, zoom

# =============================================================================
# Physical parameters (aligned with ChannelConfig / MPRAGE PSF)
# =============================================================================
SPACING_IN_XY = (218/336, 166/256)   # (X, Y) mm
SPACING_OUT_XY = (1.8, 1.8)          # target CEST spacing (mm)
SIGMA_ADD_MM_XY = (0.713, 0.713)     # MPRAGE sigma_add_mm used for probability labels (linear)
NUM_CLASSES_DEFAULT = 102

SIGMA_ADD_PX = (
    SIGMA_ADD_MM_XY[0] / SPACING_IN_XY[0],
    SIGMA_ADD_MM_XY[1] / SPACING_IN_XY[1],
)

ZOOM_FACTORS = (
    SPACING_IN_XY[0] / SPACING_OUT_XY[0],  # y-direction maps to X spacing
    SPACING_IN_XY[1] / SPACING_OUT_XY[1],  # x-direction maps to Y spacing
    1.0                                    # channels
)

# =============================================================================
# LUT helper (same column discovery style as fig1-3.ipynb)
# =============================================================================
def load_lut(file_path: str) -> dict:
    """Load label->name/color mapping from .xlsx/.csv with flexible columns."""
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        def find_col(candidates):
            for c in candidates:
                if c in df.columns:
                    return c
            return None

        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])
        col_r = find_col(["R", "r"])
        col_g = find_col(["G", "g"])
        col_b = find_col(["B", "b"])

        if (col_id is None) or (col_name is None):
            print(f"[Warn] LUT columns not found. Available: {list(df.columns)}")
            return {}

        lut = {}
        valid = pd.to_numeric(df[col_id], errors="coerce").notna()
        ids = df.loc[valid, col_id].astype(int)
        names = df.loc[valid, col_name].astype(str)
        colors = None
        if col_r and col_g and col_b:
            colors = df.loc[valid, [col_r, col_g, col_b]].astype(float).values

        for idx, name in zip(ids, names):
            entry = {"name": name}
            if colors is not None:
                vec = colors[len(lut)]
                entry.update({"R": vec[0], "G": vec[1], "B": vec[2]})
            lut[int(idx)] = entry

        print(f"Loaded {len(lut)} labels from LUT.")
        return lut
    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}

# =============================================================================
# Core processing: one-hot -> PSF smoothing -> linear downsample
# =============================================================================
def _normalize_prob(p: np.ndarray) -> np.ndarray:
    s = p.sum(axis=-1, keepdims=True)
    s = np.where(s > 0, s, 1.0)
    return p / s

def build_color_table(lut_dict: dict, num_classes: int) -> np.ndarray:
    cmap_fallback = plt.cm.tab20
    colors = np.zeros((num_classes, 3), dtype=float)
    for i in range(num_classes):
        entry = lut_dict.get(i)
        if isinstance(entry, dict):
            r = entry.get("R") or entry.get("r")
            g = entry.get("G") or entry.get("g")
            b = entry.get("B") or entry.get("b")
            if r is not None and g is not None and b is not None:
                vec = np.array([r, g, b], dtype=float)
                colors[i] = vec / (255.0 if vec.max() > 1.0 else 1.0)
                continue
            if "color" in entry:
                vec = np.array(entry["color"], dtype=float)
                colors[i] = vec[:3] / (255.0 if vec.max() > 1.0 else 1.0)
                continue
        elif isinstance(entry, (list, tuple)) and len(entry) >= 3:
            vec = np.array(entry[:3], dtype=float)
            colors[i] = vec / (255.0 if vec.max() > 1.0 else 1.0)
            continue
        colors[i] = np.array(cmap_fallback(i % cmap_fallback.N)[:3])
    return colors

def lut_name(label: int, lut_dict: dict) -> str:
    entry = lut_dict.get(int(label))
    if isinstance(entry, dict):
        for k in ("freesurfer_tissue_name", "tissue_name", "name", "label"):
            if k in entry:
                return str(entry[k])
    return str(entry) if entry is not None else f"Label {label}"

def one_hot_encode(labels_2d: np.ndarray, num_classes: int) -> np.ndarray:
    h, w = labels_2d.shape
    one_hot = np.zeros((h, w, num_classes), dtype=np.float32)
    for c in range(num_classes):
        mask = (labels_2d == c)
        if mask.any():
            one_hot[..., c] = mask.astype(np.float32)
    return one_hot

def process_roi(roi_labels: np.ndarray, num_classes: int | None = None):
    """One-hot -> anisotropic Gaussian (xy) -> linear downsample to target grid."""
    max_label = int(np.max(roi_labels))
    num_classes = num_classes or max(NUM_CLASSES_DEFAULT, max_label + 1)

    one_hot = one_hot_encode(roi_labels, num_classes)

    soft_hr = gaussian_filter(one_hot,
                              sigma=(SIGMA_ADD_PX[0], SIGMA_ADD_PX[1], 0.0),
                              mode="constant")
    soft_hr = _normalize_prob(soft_hr)

    soft_lr = zoom(soft_hr, zoom=ZOOM_FACTORS, order=1, mode="reflect")
    soft_lr = _normalize_prob(soft_lr)

    return soft_hr.astype(np.float32), soft_lr.astype(np.float32), num_classes


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Visualization helpers
"""

def pick_mixed_voxel(proba_lr: np.ndarray):
    entropy = -np.sum(proba_lr * np.log(proba_lr + 1e-8), axis=-1)
    y, x = np.unravel_index(np.argmax(entropy), entropy.shape)
    return int(y), int(x), proba_lr[y, x]

def prob_to_rgb(prob_map: np.ndarray, colors: np.ndarray) -> np.ndarray:
    return np.tensordot(prob_map, colors, axes=([2], [0]))

def plot_figure_5_1(roi_anatomy: np.ndarray, roi_labels: np.ndarray, lut_dict: dict):
    assert roi_anatomy.shape == roi_labels.shape, "roi_anatomy and roi_labels must align"

    soft_hr, soft_lr, num_classes = process_roi(roi_labels)
    colors = build_color_table(lut_dict, num_classes)

    h_hr, w_hr = roi_labels.shape
    width_mm = w_hr * SPACING_IN_XY[1]   # width corresponds to Y spacing
    height_mm = h_hr * SPACING_IN_XY[0]  # height corresponds to X spacing

    rgb_hr = prob_to_rgb(soft_hr, colors)
    rgb_lr = prob_to_rgb(soft_lr, colors)

    vy, vx, voxel_prob = pick_mixed_voxel(soft_lr)
    h_lr, w_lr, _ = soft_lr.shape
    voxel_w_mm = width_mm / w_lr
    voxel_h_mm = height_mm / h_lr

    fig = plt.figure(figsize=(18, 4.8), facecolor="white")
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 1.05], wspace=0.18)

    # Panel A: High-res hard labels
    axA = fig.add_subplot(gs[0])
    axA.imshow(roi_anatomy, cmap="gray", origin="upper")
    axA.imshow(roi_labels, cmap=ListedColormap(colors), origin="upper", alpha=0.55,
               interpolation="nearest", vmin=0, vmax=num_classes-1)
    axA.set_title("A. High-Res Hard Labels\n(0.65mm Isotropic)", fontweight="bold")
    axA.axis("off")

    # Panel B: PSF smoothing (probability blend)
    axB = fig.add_subplot(gs[1])
    axB.imshow(roi_anatomy, cmap="gray", origin="upper", alpha=0.25)
    axB.imshow(rgb_hr, origin="upper")
    axB.set_title("B. Physical Smoothing\n(Simulating PSF)", fontweight="bold")
    axB.axis("off")

    # Panel C: Native CEST grid (pixelated) + grid lines + cyan box
    axC = fig.add_subplot(gs[2])
    extent = [0, width_mm, height_mm, 0]
    axC.imshow(rgb_lr, origin="upper", interpolation="nearest", extent=extent)

    for gx in np.arange(0, width_mm + 1e-6, SPACING_OUT_XY[1]):
        axC.axvline(gx, color="white", lw=0.6, alpha=0.9)
    for gy in np.arange(0, height_mm + 1e-6, SPACING_OUT_XY[0]):
        axC.axhline(gy, color="white", lw=0.6, alpha=0.9)

    rect = plt.Rectangle((vx * voxel_w_mm, vy * voxel_h_mm), voxel_w_mm, voxel_h_mm,
                         edgecolor="cyan", facecolor="none", lw=2.0)
    axC.add_patch(rect)
    axC.set_title("C. Native CEST Grid\n(1.8mm × 1.8mm)", fontweight="bold")
    axC.set_xticks([])
    axC.set_yticks([])

    # Panel D: Voxel signature (top-4 probabilities)
    axD = fig.add_subplot(gs[3])
    top_idx = np.argsort(voxel_prob)[::-1][:4]
    top_vals = voxel_prob[top_idx]
    bar_colors = colors[top_idx]
    names = [lut_name(int(i), lut_dict) for i in top_idx]

    axD.bar(range(len(top_idx)), top_vals, color=bar_colors, edgecolor="k")
    axD.set_xticks(range(len(top_idx)))
    axD.set_xticklabels(names, rotation=20, ha="right")
    axD.set_ylim(0, 1.0)
    axD.set_ylabel("Probability")
    axD.set_title("D. Soft Voxel Signature\n(Target voxel)", fontweight="bold")

    plt.tight_layout()
    return fig


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Execution: load MAT + LUT -> select slice -> crop -> PSF soft labels -> plot
"""

import h5py
from pathlib import Path

# ===== Paths (edit to your environment) =====
DATA_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat")
LUT_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")
OUTPUT_FIG = Path("Figure_5_1_per_voxel_soft_labels.png")
SLICE_IDX = 222
ANATOMY_CHANNEL = 341  # MPRAGE channel
CROP_MARGIN = 20       # pixels

def load_data_and_labels(mat_path: Path):
    print(f"Loading data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        if "data" not in f:
            raise ValueError("MAT 文件缺少 data")
        data = f["data"][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        if "region_labels" in f:
            labels = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            oh = f["one_hot_loc_alex_label"][:]
            axis = 0 if oh.shape[0] == 102 else -1
            labels = np.argmax(oh, axis=axis)
        else:
            raise ValueError("MAT 文件缺少 region_labels/one_hot_loc_alex_label")
    print(f"  data shape: {data.shape}, labels shape: {labels.shape}")
    return data.astype(np.float32), labels.astype(int)

def crop_to_nonzero(lbl2d: np.ndarray, margin: int = 0):
    ys, xs = np.nonzero(lbl2d)
    if len(ys) == 0:
        return (0, lbl2d.shape[0], 0, lbl2d.shape[1])
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    y0 = max(0, y0 - margin); y1 = min(lbl2d.shape[0], y1 + margin)
    x0 = max(0, x0 - margin); x1 = min(lbl2d.shape[1], x1 + margin)
    return (y0, y1, x0, x1)

def run_figure_5_1():
    lut_dict = load_lut(str(LUT_PATH))
    data, labels = load_data_and_labels(DATA_PATH)
    assert 0 <= SLICE_IDX < data.shape[0], "SLICE_IDX 越界"
    assert ANATOMY_CHANNEL < data.shape[-1], "ANATOMY_CHANNEL 越界"

    img_slice = data[SLICE_IDX, :, :, ANATOMY_CHANNEL]
    lbl_slice = labels[SLICE_IDX, :, :]

    y0, y1, x0, x1 = crop_to_nonzero(lbl_slice, margin=CROP_MARGIN)
    img_roi = img_slice[y0:y1, x0:x1]
    lbl_roi = lbl_slice[y0:y1, x0:x1]

    fig = plot_figure_5_1(img_roi, lbl_roi, lut_dict)
    fig.savefig(OUTPUT_FIG, dpi=300)
    plt.close(fig)
    print(f"✅ Figure saved: {OUTPUT_FIG}")

run_figure_5_1()
